# OpenPlaque — Source-Volume Coronary Centerlines

Fresh experiment from `main`. This abandons Siemens curved-reformat tracking and works directly in source CCTA series 7.

The existing validated RCA source centerline is reused by default. LAD/LCX are generated as automatic source-volume hypotheses from a left-coronary ostium and paired global graph routes. **They must pass visual anatomical QC before any plaque or non-contrast calcium work uses them.**

This notebook also generates OpenPlaque-owned rotating CPR stacks from the source-volume centerlines, preserving reconstructible geometry for future plaque mapping.


## Step 1 — Mount Google Drive

In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache / reuse controls

All default to `True`: reuse a valid cache if available; otherwise recompute and cache. Set a flag to `False` to force recomputation of that component.

In [ ]:
REUSE_SOURCE_EVIDENCE = True
REUSE_RCA_CENTERLINE = True
REUSE_LEFT_OSTIUM = True
REUSE_LEFT_BRANCHES = True
REUSE_CPR_STACKS = True
REUSE_QC_FIGURES = True
REUSE_REPORT_PACKAGE = True


## Step 3 — Install this source-volume branch

This workflow does **not** install nnU-Net. It uses only source CCTA, the validated aorta cache, and source-volume graph/image processing.

In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch source-volume-coronary-centerlines-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas

import sys
sys.path.insert(0, '/content/OpenPlaque/src')
import pandas as pd
from IPython.display import display, Image
from openplaque.source_centerline_workflow import SourceCenterlineWorkflow
print('Source-volume coronary workflow ready.')


## Step 4 — Initialize and inspect cache plan

In [ ]:
REUSE = {
    'source_evidence': REUSE_SOURCE_EVIDENCE,
    'rca_centerline': REUSE_RCA_CENTERLINE,
    'left_ostium': REUSE_LEFT_OSTIUM,
    'left_branches': REUSE_LEFT_BRANCHES,
    'cpr_stacks': REUSE_CPR_STACKS,
    'qc_figures': REUSE_QC_FIGURES,
    'report_package': REUSE_REPORT_PACKAGE,
}
wf = SourceCenterlineWorkflow('/content/drive/MyDrive/OpenPlaque', REUSE)
display(wf.cache_status())


## Step 5 — Load source CCTA + validated aorta and build/reuse 3-D coronary evidence

The source volume is series 7. The evidence volume is downsampled to about 1 mm for graph search and cached on Drive.

In [ ]:
wf.prepare_source()
evd = wf.prepare_evidence()
print('Source CCTA shape:', wf.ct.shape, 'spacing zyx:', wf.spacing_zyx)
print('Graph volume shape:', evd['ct'].shape, 'spacing zyx:', evd['spacing_zyx'])


## Step 6 — RCA source-volume centerline

Default behavior imports the previously validated frozen RCA centerline into this workflow cache. Set `REUSE_RCA_CENTERLINE=False` to force a new source-volume global-graph trace from the frozen RCA ostial seed.

In [ ]:
rca = wf.build_rca_centerline()
print('RCA points:', len(rca))


## Step 7 — Detect/reuse the left-coronary ostium

The detector searches the aortic wall opposite the RCA seed and requires coronary-scale support continuing outward from the aorta.

In [ ]:
left_seed, left_direction = wf.detect_left_ostium()
print('Left ostium graph zyx:', left_seed)
display(wf.left_candidates.head(15))


## Step 8 — Extract paired LAD / LCX global-graph routes

The pair must share a plausible short left-main trunk, then diverge substantially. Anatomical labeling uses source-volume LPS orientation: the more inferior/anterior route is labeled LAD and the other LCX.

In [ ]:
lad, lcx = wf.build_left_branches()
print('LAD points:', len(lad), 'LCX points:', len(lcx))


## Step 9 — Centerline numerical QC

In [ ]:
qc = wf.build_all_centerlines()
display(qc)


## Step 10 — Generate/reuse OpenPlaque rotating CPR stacks

Each stack contains 24 rotations about the source-volume centerline plus centerline/frame geometry needed to reconstruct the source location of every future CPR sample.

In [ ]:
cpr = wf.build_cpr_stacks()
for vessel, d in cpr.items():
    print(vessel, 'stack', d['stack'].shape, 'length mm', float(d['arc_mm'][-1]))


## Step 11 — Visual anatomical QC

Review all three figures carefully. A centerline is not accepted merely because graph scores are high.

In [ ]:
figs = wf.plot_qc()
for fp in figs:
    print(fp)
    display(Image(filename=str(fp)))


## Step 12 — Package report-back ZIP

In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_SOURCE_CENTERLINES_REPORT_BACK.zip')
print('\nCache provenance:')
display(pd.read_csv(wf.out / 'cache_provenance.csv'))
